In [ ]:
from abs_affinity_based_slotting.config import RAW_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.demand import (build_cooccurrence, build_sku_demand,
    affinity_registry, filter_registry)
from abs_affinity_based_slotting.warehouse import (occupied_locations,
    build_location_costs, build_bay_distance_matrix)
from abs_affinity_based_slotting.slotting import build_instance
from abs_affinity_based_slotting.clustering import clustering_registry
from abs_affinity_based_slotting.methods import (CurrentSlotting, DemandGreedySlotting,
    LinearAssignmentSlotting, SwapSearchSlotting, BiLevelSlotting)
from abs_affinity_based_slotting.evaluation import Evaluator
import pandas as pd, time

In [ ]:
# Cell 2 — datos + geometria fija (f, c, D, universo)
ds = WarehouseDataLoader(RAW_DIR).load_all()
split = split_picking_events(ds.picking_events, test_size=0.2)
universe   = occupied_locations(ds.initial_stock)["sku"].to_numpy()
sku_demand = build_sku_demand(split.train)  #f
loc_costs  = build_location_costs(ds.initial_stock, ds.distances) #c
bay_dist   = build_bay_distance_matrix(ds.distances) #D
evaluator  = Evaluator.from_tables(ds.coordinates, ds.distances, ds.initial_stock)

In [3]:
# Cell 3 — armado componible de la instancia: se elige A (metrica + filtro)
def make_instance(affinity="jaccard", filt=("top_k", {"k": 10})):
    co = build_cooccurrence(split.train, skus=universe)
    A = affinity_registry.get(affinity)().build(co.matrix, co.support, co.n_batches)
    if filt is not None:
        name, kw = filt
        A = filter_registry.get(name)(**kw).filter(A)
    return build_instance(sku_demand, loc_costs, bay_dist,
                          initial_stock=ds.initial_stock, skus=universe, affinity=A)

instance = make_instance()   # jaccard + top_k(10)
instance


SlottingInstance(n_skus=27000, n_locations=30000, n_bays=1001, affinity_edges=350986)

In [ ]:
# Cell 4 — helper de corrida (metodo -> metricas)
def run(name, method, instance):
    t0 = time.time(); sol = method.solve(instance); dt = time.time() - t0
    m = evaluator.evaluate(sol, split.test)
    return {"metodo": name, "mean": round(m.mean_batch_distance),
            "p95": round(m.p95_batch_distance), "seg": round(dt, 1)}


In [ ]:
# Cell 5 — barrido componible: baselines vs bi-nivel
clu = lambda n: clustering_registry.get(n)()
rows = [
    run("current", CurrentSlotting(ds.initial_stock), instance),
    run("demand_greedy", DemandGreedySlotting(), instance),
    run("bilevel[merchant, zona=linear]", BiLevelSlotting(clu("merchant"), LinearAssignmentSlotting()), instance),
    run("bilevel[merchant, zona=swap_search]", BiLevelSlotting(clu("merchant"), SwapSearchSlotting(lam=0.5)), instance),
]
pd.DataFrame(rows)
# nota: evitar bilevel[demand_class, zona=linear] -> cluster de 18k cuelga linear_assignment